In [1]:
from pathlib import Path
import warnings

import scanpy as sc
# import scib
# import numpy as np
import sys

sys.path.insert(0, "../")

import scgpt as scg
import matplotlib.pyplot as plt
# import anndata
import pandas as pd

plt.style.context('default')
warnings.simplefilter("ignore", ResourceWarning)

/scratch/2370352/conda/envs/scgpt/lib/python3.10/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/scratch/2370352/conda/envs/scgpt/lib/python3.10/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/scratch/2370352/conda/envs/scgpt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Create your set-up:

# model - choose 1:
model = "whole_human"
# model = "pancancer"

# source data file, in "data" folder
filename = "adata_TCGA-LUAD.star_tpm"

# highly variable
highly_variable = False
N_HVG = 3000

if not highly_variable:
    N_HVG = 0

In [3]:
model_dir = Path(f"/scratch/2370352/my-research/papers/scgpt/save/{model}")
smaple_data_path = f'../data/0_adata_for_scgpt/{filename}.h5ad'

adata = sc.read_h5ad(smaple_data_path)
gene_col = "gene_name"
batch_key = "sample"

In [4]:
org_adata = adata.copy()

In [5]:
# highly variable genes
if highly_variable:
    sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor='seurat_v3')
    adata = adata[:, adata.var['highly_variable']]

In [6]:
embed_adata = scg.tasks.embed_data(
    adata,
    model_dir,
    gene_col=gene_col,
    batch_size=64,
)
# attach the cell embedding to the original adata

scGPT - INFO - match 20260/20260 genes in vocabulary of size 60697.


/scratch/2370352/conda/envs/scgpt/lib/python3.10/site-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells: 100%|██████████| 9/9 [00:03<00:00,  2.56it/s]
/scratch/2370352/conda/envs/scgpt/lib/python3.10/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings


In [7]:
# zapis embeddingów do pliku

X_emb = embed_adata.obsm["X_scGPT"]

# nazwy kolumn emb1, emb2, ...
emb_cols = [f"emb{i+1}" for i in range(X_emb.shape[1])]

# sample IDs (najczęściej index obs)
samples = embed_adata.obs.index

df_emb = pd.DataFrame(
    X_emb,
    index=samples,
    columns=emb_cols
)

# przenieś index do kolumny "sample"
df_emb = df_emb.reset_index().rename(columns={"index": "Unnamed: 0"})

# zapis do CSV
df_emb.to_csv(f"../data/scgpt_embeddings/scgpt_{filename}_{model}_hvg_{N_HVG}.csv", index=False)